In [1]:
import chipwhisperer as cw
import time

# Connection
scope = cw.scope()


# Bitstream Path
# Note: Using 'r' before the string to handle backslashes correctly
bitstream = r'C:\Users\sbista\ChipWhisperer\chipwhisperer\firmware\fpgas\aes\vivado_cw\cw305_aes.runs\impl_100t\cw305_top.bit'

# Program FPGA
target = cw.target(scope, cw.targets.CW305, bsfile=bitstream, force=True)

# Set some default CW305 settings
target.vccint_set(1.0)
#target.pll.set_cw305_vccint_settings(1.0)
print("FPGA Programmed Successfully")

(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (C:\Users\sbista\ChipWhisperer\chipwhisperer\software\chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


FPGA Programmed Successfully


In [2]:
# Scope Setup
scope.gain.gain = 40
scope.gain.mode = "high"
scope.adc.samples = 350
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"

# Clock Settings
scope.clock.extclk_freq = 10000000
scope.clock.clkgen_mul = 5
scope.clock.clkgen_div = 48
scope.clock.clkgen_freq = 10000000
scope.clock.adc_src = "clkgen_x4"
scope.clock.freq_ctr_src = "clkgen"
scope.clock.adc_phase = 175

# Glitch Setup
#scope.io.hs2 = "glitch"
scope.glitch.clk_src = 'clkgen'
scope.glitch.output = "clock_xor"
#scope.glitch.trigger_src = "ext_single" # For precise glitching per-trace
scope.glitch.trigger_src = "continuous"

# Turn on Glitch Transistors
scope.io.glitch_hp = True
scope.io.glitch_lp = True

print("Hardware configured.")

Hardware configured.


In [3]:
from Crypto.Cipher import AES

# Known NIST AES-128 test vector
#fixed_key = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")
#fixed_text  = bytes.fromhex("5c692f9103b2302914d7e555e4dcee49")
fixed_text = bytes.fromhex("5C 69 2F 91 03 B2 30 29 14 D7 E5 55 E4 DC EE 49")
fixed_key = bytes.fromhex("2B 7E 15 16 28 AE D2 A6 AB F7 15 88 09 CF 4F 3C")
# Expected ciphertext
expected_output1 = AES.new(fixed_key, AES.MODE_ECB).encrypt(fixed_text)
expected_output = expected_output1.hex()
print("Expected CT:", expected_output1.hex())


Expected CT: 06f36a65e8a99ff8907b2e5e5ddd77de


In [6]:

from tqdm import notebook

# Constants from your script
#fixed_text = bytearray.fromhex("5C 69 2F 91 03 B2 30 29 14 D7 E5 55 E4 DC EE 49")
#fixed_key = bytearray.fromhex("2B 7E 15 16 28 AE D2 A6 AB F7 15 88 09 CF 4F 3C")
#expected_output = "06f36a65e8a99ff8907b2e5e5ddd77de"

# Define your scan ranges (Start, Stop, Step)
import numpy as np
widths = np.arange(0.390625, 49.0, 0.390625)
offsets = np.arange(-49.6094, 49.0, 0.390625)

target.set_key(fixed_key)

print("Starting Glitch Loop...")

for width in notebook.tqdm(widths, desc="Width"):
    for offset in offsets:
        scope.glitch.width = width
        scope.glitch.offset = offset
        
        scope.arm()
        
        # Write plaintext and wait for response
        target.simpleserial_write('p', fixed_text)
        
        ret = scope.capture()
        if ret:
            # print("Target timed out - might have crashed!")
            print(f"Timeout occurred on trace ")
            continue
            
        response = target.simpleserial_read('r', 16)
        
        # Check if response is valid/hex
        if response is None:
            continue
            
        res_hex = response.hex()
        
        # Logic to detect success
        if res_hex == expected_output:
            print(f"Normal! Width: {width}, Offset: {offset}")
            print(f"Result: {res_hex}")
        else:
            print(f"🌟 SUCCESS! Width: {width}, Offset: {offset}")
            print(f"Result: {res_hex}")
            # You can 'break' here if you only want the first successful glitch

#             # Log to CSV list
#             results_log.append({"width": width, "offset": offset, "result": status, "output": res_hex})
            
#             # Save to ChipWhisperer Project
#             project.traces.append(scope.get_last_trace(), data_in=fixed_text, data_out=response, key=fixed_key)


# # Save and Export
# project.save()

# df = pd.DataFrame(results_log)
# df.to_csv("glitch_results.csv", index=False)

# print("Done! Results saved to 'glitch_results.csv' and 'Glitch_Experiment_CW305.cwp'")

Starting Glitch Loop...


Width:   0%|          | 0/125 [00:00<?, ?it/s]

🌟 SUCCESS! Width: 0.390625, Offset: -49.6094
Result: 06f36a65e8a99cf8907b2e5e5ddd0000
Normal! Width: 0.390625, Offset: -49.218775
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -48.82815
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
🌟 SUCCESS! Width: 0.390625, Offset: -48.437525
Result: 06f36a65e8a99ff890592e5e5ddd77de
Normal! Width: 0.390625, Offset: -48.0469
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -47.656275
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -47.26565
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -46.875025
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -46.4844
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -46.093775
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -45.70315
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -45.312525
Result: 06f36a65e8a99ff8

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:793) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:793) Partial reconfiguration for offset = 0 may not work


Normal! Width: 0.390625, Offset: -0.39065000000000083
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: -2.5000000000829914e-05
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
🌟 SUCCESS! Width: 0.390625, Offset: 0.39059999999999917
Result: 06f36a65e8a99ff890590c5e5ddd77de
🌟 SUCCESS! Width: 0.390625, Offset: 0.7812249999999992
Result: 06f36a65e8a99df8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: 1.1718499999999992
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: 1.5624749999999992
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: 1.9530999999999992
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: 2.343724999999999
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: 2.734349999999999
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: 3.124974999999999
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 0.390625, Offset: 3.515599999999999
Result: 

...

Normal! Width: 48.4375, Offset: -5.078150000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -4.687525000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -4.296900000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -3.906275000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -3.515650000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -3.125025000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -2.734400000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -2.343775000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -1.9531500000000008
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -1.5625250000000008
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: -1.1719000000000008
Result: 06f36a65e8a99ff890

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:793) Partial reconfiguration for offset = 0 may not work


Normal! Width: 48.4375, Offset: -2.5000000000829914e-05
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
🌟 SUCCESS! Width: 48.4375, Offset: 0.39059999999999917
Result: 00106a65e8a99ff8000000005cdd55dc
🌟 SUCCESS! Width: 48.4375, Offset: 0.7812249999999992
Result: 04000000e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: 1.1718499999999992
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: 1.5624749999999992
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
🌟 SUCCESS! Width: 48.4375, Offset: 1.9530999999999992
Result: 04f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: 2.343724999999999
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: 2.734349999999999
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: 3.124974999999999
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: 3.515599999999999
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.4375, Offset: 3.906224999999999
Result: 06f36a65e8a

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:793) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:793) Partial reconfiguration for offset = 0 may not work


Normal! Width: 48.828125, Offset: -2.734400000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.828125, Offset: -2.343775000000001
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.828125, Offset: -1.9531500000000008
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.828125, Offset: -1.5625250000000008
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.828125, Offset: -1.1719000000000008
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.828125, Offset: -0.7812750000000008
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.828125, Offset: -0.39065000000000083
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
Normal! Width: 48.828125, Offset: -2.5000000000829914e-05
Result: 06f36a65e8a99ff8907b2e5e5ddd77de
🌟 SUCCESS! Width: 48.828125, Offset: 0.39059999999999917
Result: 04f34865e8a99ff8907b2e5e5ddd77de
🌟 SUCCESS! Width: 48.828125, Offset: 0.7812249999999992
Result: 0400000048a99fd89000000000001000
🌟 SUCCESS! Width: 48.828125, Offset: 1.171

In [ ]:
from tqdm import notebook # For a nice progress bar

# Define ranges
widths = [0.390625 + i*0.390625 for i in range(100)] # Example range
offsets = [-14.6094 + i*0.390625 for i in range(50)]

for width in notebook.tqdm(widths):
    for offset in offsets:
        scope.glitch.width = width
        scope.glitch.offset = offset
        
        # Arm scope and send plaintext
        scope.arm()
        target.simpleserial_write('p', bytearray([0x5C, 0x69...])) # Your fixed plaintext
        
        ret = scope.capture()
        if ret: continue # Skip if timeout
        
        output = target.simpleserial_read('r', 16)
        # Check for glitch success here